# Step 12 — governance documentation validation

This notebook validates the completed `SUBMISSION.md` governance documentation and performs repository safety checks. It does not edit source data, credentials, or the read-only snapshots.

In [1]:
# Resolve the repository root so the notebook works from VS Code or Jupyter.
from pathlib import Path
import re

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'SUBMISSION.md').exists():
    REPO_ROOT = Path.cwd().parent
SUBMISSION = REPO_ROOT / 'SUBMISSION.md'
DOCUMENT = SUBMISSION.read_text()
print(f'Checking: {SUBMISSION}')

Checking: /Users/kumarsamanvaya/Documents/IITR - FDE Course/Data Pipepline Assignment/_Samy_IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/SUBMISSION.md


In [2]:
# Confirm that every required governance topic is documented.
required_topics = {
    'Dataset ownership': 'Dataset ownership',
    'Data dictionary': 'Data dictionary',
    'Data lineage': 'Data lineage',
    'Personal-data handling': 'Personal-data handling',
    'Access control': 'Access control',
    'Retention': 'Retention',
    'Secrets management': 'Secrets management',
    'Auditability': 'Auditability',
    'Change history': 'Change history and returned orders',
}
missing_topics = [label for label, heading in required_topics.items() if heading.lower() not in DOCUMENT.lower()]
assert not missing_topics, f'Missing governance topics: {missing_topics}'
print(f'Governance topics present: {len(required_topics)}')

Governance topics present: 9


In [3]:
# Scan source/config/documentation files for values that look like committed
# secrets. Names of environment variables are allowed; secret values are not.
secret_patterns = [
    re.compile(r'(?i)sk-[A-Za-z0-9]{20,}'),
    re.compile(r'(?i)xox[baprs]-[A-Za-z0-9-]{20,}'),
    re.compile(r'https://hooks\.slack\.com/services/[A-Za-z0-9/_-]+'),
    # Flag long literal values, but allow code that retrieves secrets from
    # Airflow Variables or environment variables at runtime.
    re.compile(r'(?i)(api[_ -]?key|password|secret|token)\s*[:=]\s*(?!(?:Variable\.get|os\.getenv|os\.environ))[A-Za-z0-9_./-]{20,}'),
]
scan_extensions = {'.py', '.md', '.yml', '.yaml', '.json', '.toml', '.ini', '.cfg', '.env'}
secret_hits = []
for path in REPO_ROOT.rglob('*'):
    if not path.is_file() or path.suffix.lower() not in scan_extensions:
        continue
    if any(part in {'.git', '.venv', '.astro', '__pycache__'} for part in path.parts):
        continue
    text = path.read_text(errors='ignore')
    if any(pattern.search(text) for pattern in secret_patterns):
        secret_hits.append(str(path.relative_to(REPO_ROOT)))
print('Secret-shaped values found:', secret_hits or 'none')
assert not secret_hits, 'Review these files before committing: possible secret values found.'

Secret-shaped values found: none


In [4]:
# Detect customer CSVs outside the explicitly approved input-data folders.
# Quarantine files are reported because they can contain raw customer fields;
# they should be ignored or stored in a protected external location.
approved_input_dirs = {
    REPO_ROOT / 'include' / 'data' / 'nammamart',
    REPO_ROOT / 'assignment' / 'nammamart_dataset',
}
customer_data_files = []
outside_approved_inputs = []
for path in REPO_ROOT.rglob('*.csv'):
    if any(part in {'.git', '.venv', '.astro'} for part in path.parts):
        continue
    header = path.read_text(errors='ignore').splitlines()[:1]
    is_customer_data = 'customer_id' in (header[0] if header else '') and ('email' in header[0] or 'phone' in header[0])
    if not is_customer_data:
        continue
    customer_data_files.append(str(path.relative_to(REPO_ROOT)))
    if not any(path.is_relative_to(directory) for directory in approved_input_dirs):
        outside_approved_inputs.append(str(path.relative_to(REPO_ROOT)))
print('Customer-data files:', customer_data_files)
print('Outside approved input folders:', outside_approved_inputs or 'none')
if outside_approved_inputs:
    print('Action required: remove these generated quarantine copies from version control or move them to protected storage.')


Customer-data files: ['assignment/quarantine_samy_step7/customers_failed_step7-etl-increment.csv', 'assignment/quarantine_samy_step7/customers_failed_step7-etl-initial.csv', 'assignment/quarantine_samy_step9/customers_failed_step9-etl-success.csv', 'assignment/nammamart_dataset/customers.csv', 'assignment/quarantine_samy_step5/customers_failed_step5-local-rerun.csv', 'assignment/quarantine_samy_step5/customers_failed_step5-local-test.csv', 'include/data/nammamart/customers.csv', 'include/quarantine/samy/customers_failed_manual__2026-10-22T13_04_03_755198_00_00.csv', 'include/quarantine/samy/customers_failed_20261017T085303Z.csv', 'include/quarantine/samy/customers_failed_20261019T033016Z.csv', 'include/quarantine/samy/customers_failed_20261017T033015Z.csv', 'include/quarantine/samy/customers_failed_20261020T033014Z.csv', 'include/quarantine/samy/customers_failed_20261015T081001Z.csv', 'include/quarantine/samy/customers_failed_20261016T034418Z.csv', 'include/quarantine/samy/customers_fa

In [5]:
# Produce a concise compliance summary for the notebook output.
status = 'passed' if not outside_approved_inputs else 'passed with data-path review required'
print(f'Step 12 documentation validation {status}.')
print(f'Required governance sections: {len(required_topics)} present')
print(f'Secret-shaped values: {len(secret_hits)}')
print(f'Customer-data paths requiring review: {len(outside_approved_inputs)}')
print('No source files were modified; this notebook only reads and reports.')

Step 12 documentation validation passed with data-path review required.
Required governance sections: 9 present
Secret-shaped values: 0
Customer-data paths requiring review: 20
No source files were modified; this notebook only reads and reports.
